In [3]:
import cv2
import numpy as np
import random

In [4]:
def add_stone_texture(shape, intensity=20):
    """Generate stone-like background texture"""
    noise = np.random.normal(128, intensity, shape).astype(np.uint8)
    noise = cv2.GaussianBlur(noise, (31, 31), 0)
    return noise

def uneven_illumination(img):
    """Apply uneven lighting"""
    h, w = img.shape
    grad_x = np.tile(np.linspace(0.8, 1.2, w), (h, 1))
    grad_y = np.tile(np.linspace(1.2, 0.8, h), (w, 1)).T
    illum = grad_x * grad_y
    return np.clip(img * illum, 0, 255).astype(np.uint8)

def stroke_erosion(binary):
    """Simulate erosion of carved strokes"""
    k = random.choice([2, 3])
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
    return cv2.erode(binary, kernel, iterations=1)

def add_carving_noise(img, amount=0.02):
    """Salt-like noise to imitate stone cracks"""
    noisy = img.copy()
    num = int(amount * img.size)
    coords = (
        np.random.randint(0, img.shape[0], num),
        np.random.randint(0, img.shape[1], num)
    )
    noisy[coords] = np.random.randint(0, 80)
    return noisy

# ---------- Main Transformation ----------

def brahmi_to_inscription(img_path, out_path):
    # Load grayscale
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    assert img is not None, "Image load failed"

    # Binarize
    _, binary = cv2.threshold(img, 150, 255, cv2.THRESH_BINARY_INV)

    # Stroke erosion (wear)
    binary = stroke_erosion(binary)

    # Blur edges slightly
    binary = cv2.GaussianBlur(binary, (3, 3), 0)

    # Stone texture background
    stone_bg = add_stone_texture(img.shape)

    # Combine carving with stone
    carved = cv2.subtract(stone_bg, binary)

    # Uneven illumination
    carved = uneven_illumination(carved)

    # Add carving noise
    carved = add_carving_noise(carved)

    # Slight motion blur (irregular depth)
    k = random.choice([3, 5])
    carved = cv2.GaussianBlur(carved, (k, k), 0)

    # Final contrast normalization
    carved = cv2.normalize(carved, None, 0, 255, cv2.NORM_MINMAX)

    cv2.imwrite(out_path, carved)

In [5]:
if __name__ == "__main__":
    brahmi_to_inscription(
        "data/imgs/unnoised/1.jpg",
        "outputs/brahmi_inscription_like.jpg"
    )

In [1]:
import cv2
import numpy as np
from skimage.util import random_noise

def simulate_inscription(
    img_path,
    output_path,
    erosion_iter=1,
    distortion_strength=5,
    noise_amount=0.02
):
    # Load image
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

    # Normalize
    img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX)

    # Binarize (text assumed black on white)
    _, binary = cv2.threshold(
        img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )

    # ---- Edge distortion (stone carving irregularity) ----
    h, w = binary.shape
    dx = (np.random.rand(h, w) - 0.5) * distortion_strength
    dy = (np.random.rand(h, w) - 0.5) * distortion_strength

    x, y = np.meshgrid(np.arange(w), np.arange(h))
    map_x = (x + dx).astype(np.float32)
    map_y = (y + dy).astype(np.float32)

    distorted = cv2.remap(
        binary, map_x, map_y, interpolation=cv2.INTER_LINEAR
    )

    # ---- Erosion to simulate chipped carving ----
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    carved = cv2.erode(distorted, kernel, iterations=erosion_iter)

    # ---- Stone texture generation ----
    stone = np.random.normal(180, 25, (h, w)).astype(np.uint8)
    stone = cv2.GaussianBlur(stone, (21, 21), 0)

    # ---- Combine carving with stone texture ----
    carved_normalized = carved / 255.0
    combined = stone * (1 - 0.6 * carved_normalized)

    # ---- Uneven illumination (age/weathering) ----
    gradient = np.linspace(0.85, 1.05, w)
    illumination = np.tile(gradient, (h, 1))
    combined = combined * illumination

    # ---- Add noise (surface degradation) ----
    combined = random_noise(
        combined / 255.0,
        mode='gaussian',
        var=noise_amount
    )
    combined = (combined * 255).astype(np.uint8)

    # ---- Final blur (aged stone softness) ----
    final = cv2.GaussianBlur(combined, (3, 3), 0)

    cv2.imwrite(output_path, final)


In [2]:
simulate_inscription(
    img_path="data/imgs/unnoised/1.jpg",
    output_path="outputs/brahmi_inscription.jpg",
    erosion_iter=2,
    distortion_strength=6,
    noise_amount=0.03
)

In [142]:
import cv2
import numpy as np
import random

def generate_stone_texture(shape):
    """Generates a synthetic noisy stone texture."""
    # Base gray noise
    noise = np.random.normal(128, 20, shape).astype(np.uint8)
    
    # Apply Gaussian blur to create 'clumps' (structure) rather than just static
    structure = cv2.GaussianBlur(noise, (5, 5), 0)
    
    # Add high-frequency grit (salt and pepper style)
    grit = np.random.randint(0, 255, shape, dtype=np.uint8)
    grit_mask = np.random.rand(*shape) < 0.1 # 10% grit coverage
    
    # Blend
    texture = cv2.addWeighted(structure, 0.7, noise, 0.3, 0)
    texture[grit_mask] = cv2.addWeighted(texture, 0.5, grit, 0.5, 0)[grit_mask]
    
    return texture

def apply_erosion(mask):
    """Randomly erodes the text to simulate weathering."""
    kernel_size = 2#random.choice([3, 5])
    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    
    # Randomly dilate (thicken) or erode (thin) to vary font weight
    # if random.random() > 0.5:
    return cv2.erode(mask, kernel, iterations=1)
    # else:
    #     # Sometimes inscriptions are thick but worn
    #     dilated = cv2.dilate(mask, kernel, iterations=1)
    #     # Add random holes to the thick text
    #     noise = np.random.rand(*mask.shape)
    #     dilated[noise > 0.95] = 0
    #     return dilated

def apply_engraving_effect(bg, text_mask):
    """
    Simulates depth by darkening the text area and adding a slight 
    highlight on edges to look like a carving.
    """
    # Soften the mask to simulate worn edges
    soft_mask = cv2.GaussianBlur(text_mask, (3, 3), 0)
    
    # Invert mask for calculations (0=Text, 255=Bg) if needed, 
    # but here we assume Input Mask: 255=Text, 0=Bg.
    
    # 1. Darken the stone where the text is (Shadow/Depth)
    # We create a 'shadow' layer
    carved = bg.astype(np.float32)
    shadow_intensity = random.randint(40, 60)
    
    # Normalize mask to 0-1 for blending
    alpha = soft_mask.astype(np.float32) / 255.0
    
    # Apply shadow: result = bg * (1 - alpha) + (bg - intensity) * alpha
    # Simplified: Subtract intensity where alpha is high
    carved -= (alpha * shadow_intensity)
    
    # 2. Add slight highlight on one side (simulating light source angle)
    # Shift mask slightly to create a 'rim'
    rows, cols = bg.shape
    M = np.float32([[1, 0, 2], [0, 1, 2]]) # Shift 2px right and down
    highlight_mask = cv2.warpAffine(soft_mask, M, (cols, rows))
    
    # The highlight appears where the shifted mask is, but the original text isn't
    # (The edge of the carving)
    highlight = cv2.subtract(highlight_mask, soft_mask)
    highlight_alpha = highlight.astype(np.float32) / 255.0
    
    carved += (highlight_alpha * 40) # Add brightness
    
    return np.clip(carved, 0, 255).astype(np.uint8)

def apply_uneven_lighting(image):
    """Adds a gradient overlay to simulate shadows/sunlight."""
    rows, cols = image.shape
    # Create a gradient mask
    mask = np.zeros((rows, cols), dtype=np.float32)
    
    # Random direction for light
    if random.random() > 0.5:
        for i in range(rows):
            mask[i, :] = i / rows # Vertical gradient
    else:
        for j in range(cols):
            mask[:, j] = j / cols # Horizontal gradient
            
    # Scale mask to influence intensity
    mask = (mask * 0.4) + 0.6 # Light drops to 60% intensity at edge
    
    lit_image = image.astype(np.float32) * mask
    return lit_image.astype(np.uint8)

def synthesize_inscription(image_path, output_path):
    # 1. Load Image
    # Assumes input is black text on white bg, or binary.
    src = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if src is None:
        print(f"Error loading {image_path}")
        return

    # 2. Preprocessing to get a clean Binary Mask (255 = Text, 0 = Bg)
    # If input is black text on white bg, invert it.
    _, binary = cv2.threshold(src, 127, 255, cv2.THRESH_BINARY_INV)
    
    # 3. Create Stone Base
    stone_bg = generate_stone_texture(binary.shape)
    
    # 4. Modify Text (Erosion/Weathering)
    worn_mask = apply_erosion(binary)
    
    # 5. Carve Text into Stone
    inscription = apply_engraving_effect(stone_bg, worn_mask)
    
    # 6. Post-Processing (Lighting & Blur)
    final_result = apply_uneven_lighting(inscription)
    
    # Optional: Slight final blur to simulate focus issues/age
    final_result = cv2.GaussianBlur(final_result, (3, 3), 0)

    # Save
    cv2.imwrite(output_path, final_result)
    print(f"Saved synthetic inscription to {output_path}")

# Example Usage
# synthesize_inscription('brahmi_sample.jpg', 'brahmi_inscription_output.jpg')
# Example Usage
synthesize_inscription('data/imgs/unnoised/1.jpg', 'outputs/brahmi_inscription_output_.jpg')

Saved synthetic inscription to outputs/brahmi_inscription_output_.jpg


In [145]:
import os

def process_directory(input_dir, output_dir):
    """Process all images in a directory."""
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    for filename in os.listdir(input_dir):
        input_path = os.path.join(input_dir, filename)
        output_path = os.path.join(output_dir, filename)
        
        if os.path.isfile(input_path):
            synthesize_inscription(input_path, output_path)

# Process train and test directories
# process_directory("data/yolo/yolo_format_det/images/train", "data/yolo/yolo_format_engr/images/train")
process_directory("data/yolo/yolo_format_det/images/val", "data/yolo/yolo_format_engr/images/val")

Saved synthetic inscription to data/yolo/yolo_format_engr/images/val/0.jpg
Saved synthetic inscription to data/yolo/yolo_format_engr/images/val/1.jpg
Saved synthetic inscription to data/yolo/yolo_format_engr/images/val/11.jpg
Saved synthetic inscription to data/yolo/yolo_format_engr/images/val/125.jpg
Saved synthetic inscription to data/yolo/yolo_format_engr/images/val/126.jpg
Saved synthetic inscription to data/yolo/yolo_format_engr/images/val/127.jpg
Saved synthetic inscription to data/yolo/yolo_format_engr/images/val/136.jpg
Saved synthetic inscription to data/yolo/yolo_format_engr/images/val/143.jpg
Saved synthetic inscription to data/yolo/yolo_format_engr/images/val/146.jpg
Saved synthetic inscription to data/yolo/yolo_format_engr/images/val/148.jpg
Saved synthetic inscription to data/yolo/yolo_format_engr/images/val/158.jpg
Saved synthetic inscription to data/yolo/yolo_format_engr/images/val/160.jpg
Saved synthetic inscription to data/yolo/yolo_format_engr/images/val/167.jpg
Save